In [3]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 (复用 Exp 16 逻辑) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=1.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 这里我们将传入 'uniform'
        self.total_epochs = epochs
        
        # 依然需要识别 Head，但如果是 uniform 策略，这个其实不影响计算
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 激进一点：为了救 Crazing，我们可以让后期的噪声变得更小
        # 从 1.0 降到 0.1 (之前是 0.2)
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=1.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 1")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=1.0")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_different_epsilon',
        'name': '1_epsilon=1',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp 1
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=1.0
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_sc

In [4]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 (复用 Exp 16 逻辑) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=5.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 这里我们将传入 'uniform'
        self.total_epochs = epochs
        
        # 依然需要识别 Head，但如果是 uniform 策略，这个其实不影响计算
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 激进一点：为了救 Crazing，我们可以让后期的噪声变得更小
        # 从 1.0 降到 0.1 (之前是 0.2)
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=5.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 1")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=5.0")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_different_epsilon',
        'name': '2_epsilon=5',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp 1
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=5.0
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_sc

In [5]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 (复用 Exp 16 逻辑) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=20.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 这里我们将传入 'uniform'
        self.total_epochs = epochs
        
        # 依然需要识别 Head，但如果是 uniform 策略，这个其实不影响计算
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 激进一点：为了救 Crazing，我们可以让后期的噪声变得更小
        # 从 1.0 降到 0.1 (之前是 0.2)
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=20.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 3")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=5.0")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_different_epsilon',
        'name': '3_epsilon=20',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp 3
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=5.0
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_sc

In [6]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 (复用 Exp 16 逻辑) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=50.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 这里我们将传入 'uniform'
        self.total_epochs = epochs
        
        # 依然需要识别 Head，但如果是 uniform 策略，这个其实不影响计算
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 激进一点：为了救 Crazing，我们可以让后期的噪声变得更小
        # 从 1.0 降到 0.1 (之前是 0.2)
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=50.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 4")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=50")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_different_epsilon',
        'name': '4_epsilon=50',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp 4
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone), epsilon=50
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_sca